# 06. A LightGBM point forecast

**Where we are:** notebook 05 built the validation harness. Simple baselines scored MAE 1,876 MW
against NESO's 605 MW, and notebook 04 showed the missing piece is mostly the weather.

**What this notebook does:**

1. Builds one feature table for every half-hour, following the cut-off rules from notebook 04.
2. Trains LightGBM through the same 24 validation folds and compares a few choices.
3. Checks the chosen model on *forecast* weather, the honest comparison with NESO.

**Why LightGBM:** gradient-boosted trees handle a mix of calendar codes, lags and weather
without much preparation, learn interactions such as "midday in summer" on their own, and train
in seconds. They are the usual strong baseline for tabular forecasting, and the method most
entries in the GEFCom energy forecasting competitions built on.

**Background reading**
- [LightGBM documentation](https://lightgbm.readthedocs.io/en/stable/)
- [Hong et al., Probabilistic energy forecasting: GEFCom2014 and beyond](https://doi.org/10.1016/j.ijforecast.2016.02.001)


## Step 1. Set up and load


In [1]:
import time
from pathlib import Path

import holidays
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROCESSED = Path("../data/processed")
TEST_START = pd.Timestamp("2024-06-01", tz="Europe/London")

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({
    "figure.dpi": 90,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e4e0",
    "lines.linewidth": 2,
})

demand = pd.read_parquet(PROCESSED / "demand.parquet")
indo = pd.read_parquet(PROCESSED / "indo.parquet")
bench = pd.read_parquet(PROCESSED / "neso_benchmark.parquet")
weather_actual = pd.read_parquet(PROCESSED / "weather_actual.parquet")
weather_forecast = pd.read_parquet(PROCESSED / "weather_forecast.parquet")
print("lightgbm", lgb.__version__)


lightgbm 4.7.0


## Step 2. The feature table

**What:** one row per half-hour from 2019, with the target (`ND`), NESO's forecast and outturn
where they exist, and the features below. Everything except the weather is the same in both
versions of the table; the weather comes from ERA5 in one and from day-ahead forecasts in the other.

| Group | Features | Known at the origin because |
|---|---|---|
| Calendar | half-hour of the local day, weekday, month, day of year | it is the calendar |
| Special days | bank holiday, days from Christmas (20 Dec to 7 Jan), lockdown | fixed in advance |
| Demand lags | `INDO` at the same local time 7 and 14 days before, and on the holiday-aware similar day (notebook 05) | published by 08:00 UTC on D−1 |
| Recent level | mean `INDO` over the latest 24 hours known, and its ratio to the week before | same |
| Weather | temperature, the target day's mean temperature, wind speed, solar radiation | forecasts in testing |
| Solar | NESO's embedded solar capacity from 28 days before, and radiation × capacity | published with a delay |

**Lockdowns:** England's stay-at-home periods were 23 March to 30 June 2020 (easing through
June), 5 November to 2 December 2020 and 6 January to 28 March 2021. The flag lets the model
treat those days as unusual; it is zero for every validation and test day.


In [2]:
LOCKDOWNS = [("2020-03-23", "2020-06-30"), ("2020-11-05", "2020-12-02"), ("2021-01-06", "2021-03-28")]
ORIGIN_TIME = pd.Timedelta(hours=8, minutes=30)
bank_holidays = set(pd.to_datetime(list(holidays.UK(subdiv="ENG", years=range(2018, 2028)))))


def local_naive(utc):
    return utc.dt.tz_convert("Europe/London").dt.tz_localize(None)


indo_by_local = (indo.assign(local=local_naive(indo["start_utc"]))
                 .drop_duplicates("local").set_index("local")["INDO"].sort_index())
indo_by_utc = indo.set_index("start_utc")["INDO"].sort_index()


def demand_days_before(rows, days):
    keys = rows["local"] - pd.to_timedelta(np.broadcast_to(days, len(rows)), unit="D")
    values = indo_by_local.reindex(keys.to_numpy())
    return pd.Series(values.to_numpy(), index=rows.index).ffill().bfill().to_numpy()


def reference_days_back(target):
    if target in bank_holidays:
        back = 2
        while (target - pd.Timedelta(days=back)).dayofweek != 6:
            back += 1
        return back
    return 14 if (target - pd.Timedelta(days=7)) in bank_holidays else 7


def latest_level(target):
    """Mean INDO over the 48 periods ending 08:00 UTC on the day before, and over the week before that."""
    last_start = (target - pd.Timedelta(days=1) + ORIGIN_TIME).tz_localize("UTC") - pd.Timedelta(hours=1)
    window = pd.date_range(end=last_start, periods=48, freq="30min")
    return indo_by_utc.reindex(window).mean(), indo_by_utc.reindex(window - pd.Timedelta(days=7)).mean()


def christmas_days(dates):
    year = np.where(dates.dt.month == 1, dates.dt.year - 1, dates.dt.year)
    offset = (dates - pd.to_datetime(pd.Series(year, index=dates.index).astype(str) + "-12-25")).dt.days
    return offset.where(offset.between(-5, 13))


In [3]:
rows = demand[["start_utc", "Date", "Settlement_Period", "ND", "EMBEDDED_SOLAR_CAPACITY"]].merge(
    bench[["start_utc", "Date", "Settlement_Period", "Demand_Forecast", "Demand_Outturn"]],
    on=["start_utc", "Date", "Settlement_Period"], how="outer")
rows = rows[rows["Date"] >= "2019-01-01"].sort_values("start_utc").reset_index(drop=True)
rows["date"] = pd.to_datetime(rows["Date"])
rows["local"] = local_naive(rows["start_utc"])

calendar = pd.DataFrame(index=rows.index)
calendar["slot"] = rows["local"].dt.hour * 2 + rows["local"].dt.minute // 30
calendar["weekday"] = rows["date"].dt.dayofweek
calendar["month"] = rows["date"].dt.month
calendar["day_of_year"] = rows["date"].dt.dayofyear
calendar["holiday"] = rows["date"].isin(bank_holidays).astype(int)
calendar["christmas"] = christmas_days(rows["date"])
calendar["lockdown"] = 0
for start, end in LOCKDOWNS:
    calendar.loc[rows["date"].between(start, end), "lockdown"] = 1

lags = pd.DataFrame(index=rows.index)
lags["lag7"] = demand_days_before(rows, 7)
lags["lag14"] = demand_days_before(rows, 14)
days = pd.Series(rows["date"].unique())
back = dict(zip(days, days.map(reference_days_back)))
lags["similar_day"] = demand_days_before(rows, rows["date"].map(back).to_numpy())
levels = {d: latest_level(d) for d in days}
lags["level24"] = rows["date"].map(lambda d: levels[d][0])
lags["level_ratio"] = lags["level24"] / rows["date"].map(lambda d: levels[d][1])

capacity = demand.groupby("Date")["EMBEDDED_SOLAR_CAPACITY"].max()
capacity.index = pd.to_datetime(capacity.index) + pd.Timedelta(days=28)
solar_capacity = capacity.reindex(pd.date_range(capacity.index.min(), rows["date"].max())).ffill()

base = pd.concat([rows[["start_utc", "Date", "ND", "Demand_Forecast", "Demand_Outturn"]], calendar, lags], axis=1)
base["solar_capacity"] = rows["date"].map(solar_capacity).to_numpy()
print(base.shape, base["Date"].min(), "to", base["Date"].max())
base[["lag7", "similar_day", "level24", "solar_capacity"]].isna().mean().round(4)


(137760, 18) 2019-01-01 to 2026-11-09


lag7              0.0000
similar_day       0.0000
level24           0.0007
solar_capacity    0.0000
dtype: float64

In [4]:
def add_weather(base, weather):
    table = base.merge(weather, on="start_utc", how="left")
    local_date = local_naive(table["start_utc"]).dt.date
    table["temp_day_mean"] = table.groupby(local_date)["temperature_2m"].transform("mean")
    table["radiation_x_capacity"] = table["shortwave_radiation"] * table["solar_capacity"] / 1000
    return table


features_actual = add_weather(base, weather_actual)
features_forecast = add_weather(base, weather_forecast)
features_actual.to_parquet(PROCESSED / "features_actual.parquet", index=False)
features_forecast.to_parquet(PROCESSED / "features_forecast.parquet", index=False)

WEATHER = ["temperature_2m", "temp_day_mean", "wind_speed_10m", "shortwave_radiation", "radiation_x_capacity"]
FEATURES = list(calendar.columns) + list(lags.columns) + ["solar_capacity"] + WEATHER
print(len(FEATURES), "features:", FEATURES)


18 features: ['slot', 'weekday', 'month', 'day_of_year', 'holiday', 'christmas', 'lockdown', 'lag7', 'lag14', 'similar_day', 'level24', 'level_ratio', 'solar_capacity', 'temperature_2m', 'temp_day_mean', 'wind_speed_10m', 'shortwave_radiation', 'radiation_x_capacity']


**Found:** the table has 137,760 half-hours from 2019 to November 2026 and 18 features. The lags,
similar-day values and solar capacity are complete; the latest-level feature is missing for 0.07%
of rows (gaps in `INDO`), which LightGBM handles as missing values. The table is saved twice,
once with ERA5 weather and once with forecast weather, so notebooks 07 and 08 use exactly the
same features.


## Step 3. Training through the folds

**What:** for each of the 24 folds, train on every half-hour from 2019 up to the fold's training
cut-off, then predict the fold. Same folds and scoring as notebook 05.

**Settings:** 600 trees, learning rate 0.05, 63 leaves, at least 50 rows per leaf, and 80% of
rows and features sampled per tree. These are common starting values; I did not tune them,
because tuning on the validation folds would make the validation score optimistic, and the
choices below matter more.

**Three ways to frame the target:**
- **Direct:** predict `ND`.
- **Relative to the latest level:** predict `ND` minus the mean demand of the latest known 24
  hours, then add it back. Trees cannot predict values outside the range they were trained on,
  and notebook 03 found the level of demand falling year by year. Predicting the difference from
  the current level lets the level come from the data rather than from the trees.
- **Correction to the similar-day baseline:** predict `ND` minus notebook 05's best baseline
  (the similar day, rescaled by the latest level), then add it back. The baseline already has the
  right daily shape and level, so the trees only have to learn what the weather and calendar
  change about it.


In [5]:
FOLDS = pd.DataFrame([{"fold": f"{s:%Y-%m}", "train_until": s - pd.Timedelta(days=2),
                        "start": s, "end": s + pd.offsets.MonthEnd(0)}
                       for s in pd.date_range("2022-06-01", "2024-05-01", freq="MS")])
PARAMS = dict(n_estimators=600, learning_rate=0.05, num_leaves=63, min_child_samples=50,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, n_jobs=4, verbose=-1, random_state=0)


def offset(frame, kind):
    """The part of the forecast that does not come from the trees."""
    if kind == "level":
        return frame["level24"]
    if kind == "baseline":
        return frame["similar_day"] * frame["level_ratio"]
    return 0


def run_folds(features, train_table=None, kind="baseline", columns=FEATURES, train_years=None):
    """Train per fold on train_table (default: the same table), predict the fold's NESO rows."""
    train_table = features if train_table is None else train_table
    date = pd.to_datetime(features["Date"])
    train_date = pd.to_datetime(train_table["Date"])
    parts = []
    for fold in FOLDS.itertuples():
        keep = (train_date <= fold.train_until) & train_table["ND"].notna()
        if train_years:
            keep &= train_date > fold.train_until - pd.DateOffset(years=train_years)
        train = train_table[keep]
        model = lgb.LGBMRegressor(**PARAMS).fit(train[columns], train["ND"] - offset(train, kind))

        test = features[date.between(fold.start, fold.end) & features["Demand_Outturn"].notna()]
        parts.append(test[["start_utc", "Date", "slot", "Demand_Forecast", "Demand_Outturn"]].assign(
            forecast=model.predict(test[columns]) + offset(test, kind), fold=fold.fold))
    return pd.concat(parts, ignore_index=True), model


def scores(frame):
    error = frame["forecast"] - frame["Demand_Outturn"]
    return pd.Series({"MAE": error.abs().mean(), "RMSE": np.sqrt((error ** 2).mean()),
                      "MAPE": (error.abs() / frame["Demand_Outturn"]).mean() * 100, "bias": error.mean()})


results, models = {}, {}
FRAMINGS = {"direct": "none", "relative to level": "level", "correction to similar day": "baseline"}
for name, kind in FRAMINGS.items():
    started = time.time()
    results[name], models[name] = run_folds(features_actual, kind=kind)
    print(f"{name}: {time.time() - started:.0f} s")

neso = results["direct"].assign(forecast=results["direct"]["Demand_Forecast"])
pd.DataFrame({"NESO": scores(neso), **{n: scores(r) for n, r in results.items()}}).T.round(1)


direct: 45 s


relative to level: 48 s


correction to similar day: 49 s


,MAE,RMSE,MAPE,bias
NESO,605.3,807.7,2.3,34.1
direct,851.4,1132.0,3.3,35.4
relative to level,785.6,1057.7,3.1,31.8
correction to similar day,966.7,1292.1,3.8,45.1


**Found:**

- **Relative to the latest level wins:** MAE 786 MW, against 851 MW for predicting `ND`
  directly. Letting the level come from the latest live demand is worth 65 MW (8%), as expected
  for trees on a drifting series.
- **Correcting the similar-day baseline is worst** (967 MW). The baseline carries last week's
  weather, so the trees first have to undo last week's conditions before adding this week's.
  Starting from the latest level is a cleaner reference.
- All three are well short of NESO's 605 MW, even with actual weather.


## Step 4. What each choice is worth

**What:** rerun the best framing three more ways:

- **without weather**, to measure how much the weather features add;
- **trained on the latest two years only**, since demand has drifted (notebook 03);
- **without the lockdown flag**, to check it is not hurting.

**Why:** each answers a question from earlier notebooks with a number, on the same folds.


In [6]:
BEST = min(FRAMINGS, key=lambda name: scores(results[name])["MAE"])
print("best framing:", BEST)
variants = {
    "no weather": dict(columns=[c for c in FEATURES if c not in WEATHER]),
    "last two years only": dict(train_years=2),
    "no lockdown flag": dict(columns=[c for c in FEATURES if c != "lockdown"]),
}
for name, kwargs in variants.items():
    started = time.time()
    results[name], models[name] = run_folds(features_actual, kind=FRAMINGS[BEST], **kwargs)
    print(f"{name}: {time.time() - started:.0f} s")

table = pd.DataFrame({"NESO": scores(neso), **{n: scores(r) for n, r in results.items()}}).T
table.round(1)


best framing: relative to level


no weather: 34 s


last two years only: 31 s


no lockdown flag: 45 s


,MAE,RMSE,MAPE,bias
NESO,605.3,807.7,2.3,34.1
direct,851.4,1132.0,3.3,35.4
relative to level,785.6,1057.7,3.1,31.8
correction to similar day,966.7,1292.1,3.8,45.1
no weather,1400.8,1845.2,5.4,-61.9
last two years only,849.4,1144.2,3.3,-5.2
no lockdown flag,778.7,1055.0,3.0,43.2


**Found:**

- **Weather is worth about 615 MW of MAE** (1,401 MW without it, 786 with it). It is by far the
  most valuable group of features, as notebooks 03 to 05 suggested.
- **Training on only the last two years is worse** (849 MW). Even with the level drift, the extra
  years teach the model more about weather and holidays than they cost.
- **The lockdown flag adds nothing:** without it the model scores 779 MW, 7 MW better, which is
  within the noise of one run. All validation and test days have the flag at zero, so it can only
  matter through how it shapes the training of 2020 and 2021.

**Decision:** keep the "relative to level" framing, all years, and drop the lockdown flag. That
is the model used from here on (17 features).
